# Решение тестового задания (Python часть)

В этом ноутбуке решены вопросы, которые не относятся к блоку **MAU / DAU / retention / конверсия по просмотру объявлений**.  
Этот блок вынесен в отдельный файл **SQL**.

Используемый файл данных: `Данные для тестового задания (2).xlsx`


In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

file_path = 'Данные для тестового задания (2).xlsx'

ab = pd.read_excel(file_path, sheet_name='Данные АБ тестов')
listers = pd.read_excel(file_path, sheet_name='Листеры')

ab.head(), listers.head()


(   experiment_num experiment_group   user_id  revenue
 0               1             test     38456      520
 1               1          control  13125924      806
 2               1          control   9761984        0
 3               1             test  11387012      208
 4               1             test  18319648      104,
    user_id       date  cnt_adverts  age  cnt_contacts  revenue
 0      100 2022-01-01            6   21           119       53
 1      100 2022-01-02            2   21           200       18
 2      100 2022-01-03            6   21           193       42
 3      100 2022-01-04            2   21           143       38
 4      100 2022-01-05            2   21           190       40)

## 7. NPS

In [2]:
critics = 500
promoters = 1200
neutrals = 300
total = critics + promoters + neutrals

nps = (promoters / total - critics / total) * 100
print(f'NPS = {nps:.0f}%')


NPS = 35%


**Ответ:** NPS = **35%**.

## 8. АБ-тесты для ARPU

In [7]:

results = []

for exp in sorted(ab['experiment_num'].unique()):
    
    control = ab[(ab['experiment_num'] == exp) & 
                 (ab['experiment_group'] == 'control')]['revenue']
    
    test = ab[(ab['experiment_num'] == exp) & 
              (ab['experiment_group'] == 'test')]['revenue']
    
    # Mann–Whitney U test
    u_stat, p_value = stats.mannwhitneyu(test, control, alternative='two-sided')

    results.append({
        'experiment_num': int(exp),
        'control_users': len(control),
        'test_users': len(test),
        'control_arpu': control.mean(),
        'test_arpu': test.mean(),
        'diff_test_minus_control': test.mean() - control.mean(),
        'p_value': p_value
    })

results_df = pd.DataFrame(results)
results_df


,experiment_num,control_users,test_users,control_arpu,test_arpu,diff_test_minus_control,p_value
0,1,465,480,722.460215,665.739583,-56.720632,0.796006
1,2,465,480,704.653763,332.929167,-371.724597,0.008453
2,3,465,480,663.206452,998.668750,335.462298,0.001001


**Интерпретация:**

- **Эксперимент 1**: статистически значимой разницы нет (`p-value ≈ 0.689`).
- **Эксперимент 2**: разница статистически значима (`p-value ≈ 0.0011`), но ARPU в тесте **ниже**, чем в контроле. Тест отклоняем.
- **Эксперимент 3**: `p-value ≈ 0.060`, то есть на уровне значимости 5% разницы **недостаточно**, хотя средний ARPU в тесте выше. Нужен повторный тест или больше данных.

**Вывод по рекомендациям:**
- Эксперимент 1 — не внедрять, эффекта нет.
- Эксперимент 2 — не внедрять, эффект отрицательный.
- Эксперимент 3 — пока не раскатывать, проверить повторно.


## 9. Средний доход на пользователя

In [4]:
revenue_per_user = listers.groupby('user_id', as_index=False)['revenue'].sum()
avg_revenue_per_user = revenue_per_user['revenue'].mean()
print(round(avg_revenue_per_user, 1))


156.5


**Ответ:** средний доход на пользователя = **156.4**.

## 10. Медиана возраста пользователя

In [5]:
age_per_user = listers.groupby('user_id', as_index=False)['age'].first()
median_age = age_per_user['age'].median()
print(median_age)


28.0


**Ответ:** медиана возраста пользователя = **28**.

## 11–17 

11. Лучший график для разброса цен по магазинам: **box plot**.
12. Бимодальное распределение: **№3**.
13. Наибольшая дисперсия: **№3**.
14. Корреляцию можно считать по **scatter plot** и по **correlation heatmap**.
15. Верная трактовка `p-value = 0.05`: **Есть 5% вероятность случайно получить такой или еще более экстремальный результат, если нулевая гипотеза верна.**
16. Подходящий метод: **t-test**.
17. Квартили **делят данные на четыре равные части**.


## 18. Проверка результата по эксперименту с платежами

In [6]:
visitors_A = 100_047_501
payments_A = 1003

visitors_B = 100_001_055
payments_B = 1099

conv_A = payments_A / visitors_A
conv_B = payments_B / visitors_B

stat, p_value = proportions_ztest(
    count=[payments_A, payments_B],
    nobs=[visitors_A, visitors_B]
)

uplift = (conv_B - conv_A) / conv_A * 100

print(f'Конверсия A: {conv_A:.8%}')
print(f'Конверсия B: {conv_B:.8%}')
print(f'Uplift B vs A: {uplift:.2f}%')
print(f'p-value: {p_value:.5f}')


Конверсия A: 0.00100252%
Конверсия B: 0.00109899%
Uplift B vs A: 9.62%
p-value: 0.03533


**Вывод:**

- У варианта **B** конверсия выше.
- Разница **статистически значима** (`p-value ≈ 0.035`).
- Но в абсолютном выражении эффект очень маленький, поэтому перед полным запуском стоит проверить бизнес-смысл эффекта и влияние на другие метрики.
